In [19]:
import rasterio
from rasterio.windows import Window
import geopandas as gpd
import os
import glob
import matplotlib.pyplot as plt

In [32]:
chm_shp = "/mnt/sdc/tree_density_and_coverage/shapefiles/sahel_downloader/chm_validstion_grid_id.gpkg"
qnap_dir = "/media/rene1337/27e47104-4197-4926-a3d0-101c850014fe/landsat_degree_tile_download"
dirs = glob.glob(os.path.join(qnap_dir, "*/"))
gdf = gpd.read_file(chm_shp)
# remove nans in id_2 column
gdf = gdf[~gdf.id_2.isna()]

In [33]:
for index, row in gdf.iterrows():
    grid_id = row['id_2']
    pathname = int(grid_id.split(',')[0])
    rowname = int(grid_id.split(',')[1])
    dirname = f'{pathname+0:05}_{rowname+0:05}'
    landsat_dir = f'{qnap_dir}/{dirname}/'
    filenames = [r for r in sorted(glob.glob(f"{landsat_dir}/**/*.tif", recursive=True))]
    selected_filename = [f for f in filenames if '2012' in f][0]
    bounds = row['geometry'].bounds
    location = row['location']
    output_dir = f'/mnt/sdc/chm_ls7'
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f'{output_dir}/{location}'
    with rasterio.open(selected_filename) as src:
        rst = src.read(window=rasterio.windows.from_bounds(*bounds, transform=src.transform))
        out_meta = src.meta.copy()
        out_meta.update({
            'height': rst.shape[1],
            'width': rst.shape[2],
            'transform': rasterio.windows.transform(window=rasterio.windows.from_bounds(*bounds, transform=src.transform), transform=src.transform)
        })
        with rasterio.open(output_filename, 'w', **out_meta) as dst:
            dst.write(rst)
            
    

In [ ]:
ls7_rasters = sorted(glob.glob(f"{output_dir}/*.tif"))
ls8_rasters = sorted(glob.glob("/mnt/sdd/Senegal_chm_landsat/targets_v4/*_masked.tif"))

filtered_list = []
for ls7 in ls7_rasters:
    for ls8 in ls8_rasters:
        ls7_bn = os.path.basename(ls7)
        ls8_bn = os.path.basename(ls8)
        if ls7_bn.split('_')[0] == ls8_bn.split('_')[0]:
            filtered_list.append((ls7, ls8))   

In [49]:
import numpy as np
for ls7, ls8 in filtered_list:
    
    with rasterio.open(ls7) as src:
        ls7_data = src.read()
    with rasterio.open(ls8) as src:
        ls8_data = src.read()
    
    label = ls8_data[-1]
    
    # crop to same size
    label = label[:ls7_data.shape[1], :ls7_data.shape[2]]
    
    out_data = np.concatenate([ls7_data, np.expand_dims(label, 0)], axis=0)

    out_meta = src.meta.copy()
    out_meta.update({
        'count': out_data.shape[0]
    })
    
    with rasterio.open(f'/mnt/sdc/chm_ls7/{os.path.basename(ls7)}', 'w', **out_meta) as dst:
        dst.write(out_data)

In [50]:
ls7_rasters = sorted(glob.glob(f"{output_dir}/*.tif"))
for raster in ls7_rasters:
    rst = rasterio.open(raster).read()
    print(rst.shape)

(9, 20, 23)
(9, 16, 18)
(9, 23, 22)
(9, 24, 25)
(9, 23, 23)
(9, 29, 32)
(9, 26, 28)
(9, 28, 27)
(9, 24, 22)
(9, 18, 16)
(9, 24, 35)
(9, 19, 22)
(9, 18, 18)
(9, 18, 17)
(9, 14, 17)
(9, 21, 20)
(9, 11, 15)
(9, 14, 14)
(9, 18, 20)
(9, 19, 20)
(9, 20, 19)
(9, 17, 20)
(9, 20, 20)
(9, 21, 23)
(9, 22, 23)
(9, 18, 16)
(9, 17, 14)
(9, 17, 19)
(9, 19, 18)
(9, 18, 20)
(9, 18, 19)
(9, 21, 18)
(9, 22, 20)
(9, 21, 23)
(9, 10, 17)
